In [ ]:
import os
import gc
import time
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# ==============================================================================
# 🎛️ PARAMETER KONFIGURASI JALUR LOKAL & FILTER 
# ==============================================================================
#PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/usgs_katalog/katalog_usgs_master_2001_2025.csv'
#PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv'

PATH_CSV_FINAL = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/katalog_radar_dll/INDONESIA_STATION_INVENTORY_FINAL.csv'
OUTPUT_WAVEFORM_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/output_waveform_indonesia_0109'

LOG_ERROR_MURNI_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/log_error/error_log_murni_0109.csv'
LOG_SUCCESS_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/log_success/success_log_0109.csv'
LOG_FALLBACK_GFZ_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/log_error/failed_iris_need_gfz_0109.csv'

# ==============================================================================
# 🎚️ PARAMETER AMBANG BATAS TARGET BARU TAHUN 2006-2007 & KUOTA 200 DATA
# ==============================================================================
START_YEAR = 2004
END_YEAR = 2005
MIN_MAGNITUDE = 3.5
MAX_MAGNITUDE = 9.5

# 📌 TARGET KUOTA BARU: Maksimal mengambil 200 baris rekaman komponen per Event ID
MAX_DATA_PER_EVENT = 200 
MAX_WORKERS = 20 

ACADEMIC_USER_AGENT = (
    "ResearchProject: Doctoral Dissertation in AI and Edge Computing; "
    "Researcher: Very Kurnia Bakti (Indonesia); "
    "ID: Scopus:57209452703"
)

clients = {
    "IRIS": Client("IRIS", timeout=60, user_agent=ACADEMIC_USER_AGENT),
    "GFZ": Client("IRIS", timeout=60, user_agent=ACADEMIC_USER_AGENT)
}

COMPLETED_TASKS_SET = set()

def download_single_waveform(task_data):
    eid, time_str, eq_lat, eq_lon, net, sta, server = task_data
    task_key = f"{eid}_{net}_{sta}"
    
    if task_key in COMPLETED_TASKS_SET:
        return "SKIPPED", None
        
    try:
        t_event = UTCDateTime(time_str)
        year_folder = str(t_event.year)
        event_dir = os.path.join(OUTPUT_WAVEFORM_DIR, year_folder, str(eid))
        filename = f"{net}_{sta}_{eid}.mseed"
        file_path = os.path.join(event_dir, filename)
        
        if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
            return "SKIPPED", None
            
        cl = clients.get(server)
        if not cl:
            return "ERROR_SERVER", (eid, time_str, net, sta, server, "Server Config Error")
            
        start_time = t_event - 60
        end_time = t_event + 240
        
        # Ambil data gelombang mentah dan tampung sementara di RAM
        st = cl.get_waveforms(network=net, station=sta, location="*", channel="BH*,HH*,EH*",
                              starttime=start_time, endtime=end_time)
        
        # ==============================================================================
        # 🛡️ SENSOR PENGAMAN: VALIDASI KELENGKAPAN TRI-KOMPONEN (N, Z, E / 1, 2, Z)
        # ==============================================================================
        # Ambil huruf terakhir dari nama channel (e.g., 'BHZ' -> 'Z', 'BHN' -> 'N')
        found_channels = set([tr.stats.channel[-1].upper() for tr in st])
        
        # Definisikan subset komponen lengkap standar seismologi global
        has_standard_nze = {'Z', 'N', 'E'}.issubset(found_channels)
        has_orthogonal_z12 = {'Z', '1', '2'}.issubset(found_channels)
        
        # Jika fasa komponen tidak lengkap, gugurkan penulisan file fisik ke SSD Mac
        if not (has_standard_nze or has_orthogonal_z12):
            channels_logged = ",".join(list(found_channels))
            # Hancurkan stream dari RAM agar terhindar dari memory leak
            del st
            return "FAILED_INCOMPLETE_CHANNELS", (eid, sta, f"Missing Components (Found: {channels_logged})")
        # ==============================================================================
        
        # Jika lolos sensor kelengkapan, eksekusi pembuatan folder dan simpan biner .mseed
        os.makedirs(event_dir, exist_ok=True)
        st.write(file_path, format="MSEED")
        del st
        
        time.sleep(0.05) 
        return "SUCCESS", task_key
            
    except Exception as e:
        err_name = type(e).__name__
        if err_name in ["HTTPError", "FDSNTimeoutException", "ConnectionError", "TimeoutError"]:
            return "NEED_GFZ", (eid, time_str, net, sta, server, err_name)
        else:
            return f"FAILED_{err_name}", (eid, time_str, net, sta, server, err_name)

def build_success_index_from_storage():
    global COMPLETED_TASKS_SET
    print("🔍 Menginisialisasi Indeks Turbo Resume...")
    os.makedirs(os.path.dirname(LOG_SUCCESS_PATH), exist_ok=True)
    
    if os.path.exists(LOG_SUCCESS_PATH):
        try:
            df_succ = pd.read_csv(LOG_SUCCESS_PATH)
            COMPLETED_TASKS_SET = set(df_succ['Task_Key'].astype(str).tolist())
            print(f"✅ Berhasil memuat {len(COMPLETED_TASKS_SET):,} file sukses ke RAM Mac!")
            return
        except Exception:
            pass

    print("📂 Menyisir folder Local Disk untuk mendata file sukses...")
    scanned_keys = []
    if os.path.exists(OUTPUT_WAVEFORM_DIR):
        for root, _, files in os.walk(OUTPUT_WAVEFORM_DIR):
            for file in files:
                if file.endswith('.mseed'):
                    parts = file.replace('.mseed', '').split('_')
                    if len(parts) >= 3:
                        scanned_keys.append(f"{parts[2]}_{parts[0]}_{parts[1]}")
                        
    COMPLETED_TASKS_SET = set(scanned_keys)
    if scanned_keys:
        pd.DataFrame({"Task_Key": scanned_keys}).to_csv(LOG_SUCCESS_PATH, index=False)
    print(f"✅ Sinkronisasi Selesai! {len(COMPLETED_TASKS_SET):,} file terdata di RAM.")

def run_closest_station_pipeline():
    print(f"🛡️  Starting Fixed-Count Seismology Downloader Pipeline (Max {MAX_DATA_PER_EVENT} Data Per Event)...")
    os.makedirs(os.path.dirname(LOG_ERROR_MURNI_PATH), exist_ok=True)
    os.makedirs(os.path.dirname(LOG_FALLBACK_GFZ_PATH), exist_ok=True)
    
    build_success_index_from_storage()
    
    if not os.path.exists(PATH_CSV_FINAL) or not os.path.exists(PATH_KATALOG_MASTER):
        print("❌ Berkas peta navigasi inventory final atau katalog master tidak ditemukan!")
        return
        
    print("⏳ Loading master navigation footprint & earthquake catalog...")
    df_inventory = pd.read_csv(PATH_CSV_FINAL)
    df_master = pd.read_csv(PATH_KATALOG_MASTER)
    
    col_master_id = next((c for c in df_master.columns if 'id' in c.lower()), 'id')
    col_master_lat = next((c for c in df_master.columns if 'lat' in c.lower()), 'latitude')
    col_master_lon = next((c for c in df_master.columns if 'lon' in c.lower()), 'longitude')
    
    df_master_clean = df_master[[col_master_id, col_master_lat, col_master_lon]].copy()
    df_master_clean.columns = ['Event_ID', 'Eq_Latitude', 'Eq_Longitude']
    
    df_inventory['Event_ID'] = df_inventory['Event_ID'].astype(str)
    df_master_clean['Event_ID'] = df_master_clean['Event_ID'].astype(str)
    
    df_merged = pd.merge(df_inventory, df_master_clean, on='Event_ID', how='inner')
    
    if len(df_merged) == 0:
        print("❌ Gagal mencocokkan data! Tidak ada Event_ID yang selaras antara kedua file.")
        return

    col_time = 'Time_UTC'
    col_mag = 'Mag'
    
    df_merged[col_time] = pd.to_datetime(df_merged[col_time], errors='coerce')
    df_filtered = df_merged[
        (df_merged[col_time] >= f"{START_YEAR}-01-01") & 
        (df_merged[col_time] <= f"{END_YEAR}-12-31 23:59:59") &
        (df_merged[col_mag] >= MIN_MAGNITUDE) & 
        (df_merged[col_mag] <= MAX_MAGNITUDE)
    ].copy()
    
    if len(df_filtered) == 0:
        print("⚠️ Tidak ada data yang cocok dengan kriteria rentang target di memori.")
        return

    # Penomoran baris rekaman kumulatif alami per Event_ID (Stasiun dibebaskan)
    df_filtered['Data_Rank'] = df_filtered.groupby('Event_ID').cumcount() + 1
    
    # Saring kuota tugas secara konsisten maksimal 200 data per event gempa
    df_final_tasks = df_filtered[df_filtered['Data_Rank'] <= MAX_DATA_PER_EVENT].copy()
    
    event_counts = df_final_tasks.groupby('Event_ID').size()
    valid_eids = event_counts[event_counts >= 2].index
    df_final_tasks = df_final_tasks[df_final_tasks['Event_ID'].isin(valid_eids)]
    
    tasks = list(df_final_tasks[['Event_ID', 'Time_UTC', 'Eq_Latitude', 'Eq_Longitude', 'Net', 'Station', 'Server']].itertuples(index=False, name=None))
    total_tasks = len(tasks)
    unique_events = len(valid_eids)
    
    del df_master, df_inventory, df_master_clean, df_merged, df_filtered, df_final_tasks
    gc.collect()
    
    print(f"📊 Filter Mengunci: {unique_events:,} Kejadian Gempa Bumi Unik.")
    print(f"📡 Total Antrean Unduhan Sinyal (Maksimal {MAX_DATA_PER_EVENT} Data per Event): {total_tasks:,} Berkas Tugas.")
    
    stats = {"SUCCESS": 0, "SKIPPED": 0, "FAILED": 0, "FALLBACK": 0}
    error_logs, fallback_logs, new_success_keys = [], [], []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        pbar = tqdm(total=total_tasks, unit="file", desc="Harvesting Waveforms")
        
        for status, detail in executor.map(download_single_waveform, tasks):
            if status == "SUCCESS":
                stats["SUCCESS"] += 1
                if detail: new_success_keys.append({"Task_Key": detail})
            elif status == "SKIPPED":
                stats["SKIPPED"] += 1
            elif status == "NEED_GFZ":
                stats["FALLBACK"] += 1
                if detail: fallback_logs.append({"Event_ID": detail[0], "Time_UTC": detail[1], "Net": detail[2], "Station": detail[3], "Server": "GFZ", "Reason": detail[5]})
            else:
                stats["FAILED"] += 1
                if detail: error_logs.append({"Event_ID": detail[0], "Station": detail[1], "Error_Reason": detail[2]})
                
            pbar.update(1)
            pbar.set_postfix({"New": stats["SUCCESS"], "Skip": stats["SKIPPED"], "To_GFZ": stats["FALLBACK"], "NoData": stats["FAILED"]})
            
            total_processed = stats["SUCCESS"] + stats["SKIPPED"] + stats["FAILED"] + stats["FALLBACK"]
            if total_processed % 400 == 0:
                gc.collect()
                if new_success_keys and len(new_success_keys) >= 1000:
                    pd.DataFrame(new_success_keys).to_csv(LOG_SUCCESS_PATH, mode='a', header=not os.path.exists(LOG_SUCCESS_PATH), index=False)
                    new_success_keys.clear()
                if fallback_logs and len(fallback_logs) >= 500:
                    pd.DataFrame(fallback_logs).to_csv(LOG_FALLBACK_GFZ_PATH, mode='a', header=not os.path.exists(LOG_FALLBACK_GFZ_PATH), index=False)
                    fallback_logs.clear()

    pbar.close()
    
    if new_success_keys: pd.DataFrame(new_success_keys).to_csv(LOG_SUCCESS_PATH, mode='a', header=not os.path.exists(LOG_SUCCESS_PATH), index=False)
    if fallback_logs: pd.DataFrame(fallback_logs).to_csv(LOG_FALLBACK_GFZ_PATH, mode='a', header=not os.path.exists(LOG_FALLBACK_GFZ_PATH), index=False)
    if error_logs: pd.DataFrame(error_logs).to_csv(LOG_ERROR_MURNI_PATH, index=False)
        
    print("\n" + "="*50 + "\n🏁 PIPELINE PENGUNDUHAN KOREKSI KELOMPOK SELESAI\n" + "="*50)
    print(f"✅ Berkas Baru Sukses Terjemput      : {stats['SUCCESS']:,} file")
    print(f"🔄 Berkas Lama Aman Terlewati        : {stats['SKIPPED']:,} file")
    print(f"⚠️ Masalah Jaringan (Dialihkan GFZ) : {stats['FALLBACK']:,} file")
    print(f"❌ Berkas Gagal (Absen/Tidak Lengkap): {stats['FAILED']:,} file")
    print("="*50)

if __name__ == "__main__":
    run_closest_station_pipeline()

In [ ]:
# Cetak biru konseptual pengunduhan otomatis masa depan Anda
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import numpy as np
import h5py

def unduh_data_waveform_modern(network, station, origin_time_str):
    client = Client("IRIS")
    t_origin = UTCDateTime(origin_time_str)
    
    # 🔥 REKOMENDASI 2: Potong langsung jendela sempit sejak dari hulu server (7 detik)
    start_time = t_origin - 2  # 2 detik sebelum gempa
    end_time = t_origin + 5    # 5 detik setelah gempa (Total 7 detik)
    
    try:
        # Unduh langsung komponen vertikal (BHZ/HHZ)
        st = client.get_waveforms(network, station, "*", "?[HZ]", start_time, end_time)
        st.detrend("demean")
        st.resample(100.0) # Pastikan 100 Hz
        
        # Array akan otomatis berukuran tepat 700 sampel
        data_array = st[0].data[:700]
        
        # Jika kurang dari 700, beri padding aman
        if len(data_array) < 700:
            data_array = np.pad(data_array, (0, 700 - len(data_array)), 'constant')
            
        return data_array
    except Exception as e:
        print(f"Gagal mengunduh stasiun {station}: {str(e)}")
        return None

In [3]:
# -*- coding: utf-8 -*-
import os
import pandas as pd

# ==============================================================================
# 🎛️ CONFIGURATION PATHS (SINKRONISASI JALUR FILE SSD MAC BAPAK)
# ==============================================================================
PATH_USGS_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/usgs_katalog/katalog_usgs_master_2001_2025.csv'

# Jalur target output untuk menyimpan berkas panduan download 3C
PATH_DEMO_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/Code & Figure demo'
PATH_OUTPUT_PANDUAN = os.path.join(PATH_DEMO_DIR, 'katalog_gempa_indonesia_usgs_bmkg.csv')

# ==============================================================================
# 🎛️ PARAMETER SELEKSI GEOLOGI REGIONAL INDONESIA (BATASAN KHUSUS DISERTASI S3)
# ==============================================================================
MIN_MAGNITUDO = 4.5  # Hanya ambil gempa signifikan agar SNR gelombang seismik bagus
LAT_UTARA = 6.0      # Batas utara geografis Indonesia
LAT_SELATAN = -11.0  # Batas selatan geografis Indonesia
LON_BARAT = 95.0     # Batas barat geografis Indonesia
LON_TIMUR = 141.0    # Batas timur geografis Indonesia

def rekayasa_katalog_master_usgs():
    print("="*90)
    print("🚀 STARTING: FORENSIC DATA WRANGLING FOR INDONESIA SEISMIC CATALOG")
    print("="*90)
    
    if not os.path.exists(PATH_USGS_MASTER):
        print(f"❌ GALAT OPERASIONAL: File master USGS tidak ditemukan di jalur:\n   👉 {PATH_USGS_MASTER}")
        return
        
    print("⏳ Memuat file CSV master USGS raksasa ke dalam RAM Mac...")
    try:
        # Membaca CSV master
        df_master = pd.read_csv(PATH_USGS_MASTER)
        print(f"   ✅ Sukses Memuat: Terdeteksi total {len(df_master)} baris kejadian gempa global.")
    except Exception as e:
        print(f"❌ GALAT SAAT MEMBACA FILE CSV: {str(e)}")
        return

    # Standardisasi nama kolom menjadi huruf kecil untuk mengeliminasi variasi nama kolom
    df_master.columns = [col.lower() for col in df_master.columns]
    
    # Audit Forensik Ketersediaan Kolom Kritis Seismologi
    kolom_wajib = ['time', 'latitude', 'longitude', 'mag']
    for kol in kolom_wajib:
        if kol not in df_master.columns:
            print(f"❌ GALAT STRUKTUR DATA: Kolom '{kol}' tidak ditemukan di dalam CSV master.")
            print(f"   Kolom yang tersedia saat ini adalah: {list(df_master.columns)}")
            return

    # --------------------------------------------------------------------------
    # FASE PENYARINGAN DATA (DATA FILTERING PIPELINE)
    # --------------------------------------------------------------------------
    print("\n⏳ Mengeksekusi pipa penyaringan spasial dan magnitudo regional Indonesia...")
    
    # 1. Filter wilayah geografis Indonesia (Bounding Box)
    kondisi_lat = (df_master['latitude'] >= LAT_SELATAN) & (df_master['latitude'] <= LAT_UTARA)
    kondisi_lon = (df_master['longitude'] >= LON_BARAT) & (df_master['longitude'] <= LON_TIMUR)
    
    # 2. Filter kekuatan gempa
    kondisi_mag = df_master['mag'] >= MIN_MAGNITUDO
    
    # Gabungkan seluruh filter ke dalam sub-dataframe
    df_indonesia = df_master[kondisi_lat & kondisi_lon & kondisi_mag].copy()
    print(f"   ✅ Filter Berhasil: Ditemukan {len(df_indonesia)} kejadian gempa berkekuatan M >= {MIN_MAGNITUDO} di wilayah Indonesia.")
    
    if len(df_indonesia) == 0:
        print("⚠️ PERINGATAN: Tidak ada gempa yang lolos filter. Coba turunkan batasan MIN_MAGNITUDO.")
        return

    # --------------------------------------------------------------------------
    # FASE REKAYASA FORMAT AI-READY (DATA TRANSFORMATION)
    # --------------------------------------------------------------------------
    print("\n⏳ Mentransformasikan parameter data ke format kaku ObsPy ISO-8601...")
    
    # Standardisasi penamaan format tanggal dan waktu
    # USGS biasanya menyimpan string waktu seperti '2004-12-26T00:58:53.450Z'
    # Kita bersihkan agar ramah dibaca oleh fungsi UTCDateTime ObsPy
    df_indonesia['origin_time'] = df_indonesia['time'].str.replace('Z', '').str.split('.').str[0]

    # Injeksi Jaringan dan Stasiun Target Otomatis
    # Untuk contoh awal replikasi, kita tembak stasiun legendaris GSN IRIS di Indonesia 
    # yang terbukti aktif di rentang katalog Anda (Era Gempa Aceh/Nias 2004-2005)
    df_indonesia['network'] = 'II'
    df_indonesia['station'] = 'KAPI'  # Stasiun Kappang, Sulawesi, Indonesia

    # Pilih hanya kolom kritis yang dibutuhkan oleh Multi-threaded 3C Downloader kita kemarin
    df_final_report = df_indonesia[['network', 'station', 'origin_time']].copy()
    
    # Ubah susunan nama kolom agar serasi (Capital Letter Format)
    df_final_report.columns = ['Network', 'Station', 'Origin_Time']
    
    # Hapus baris duplikat jika ada untuk mencegah redundansi download data
    df_final_report.drop_duplicates(inplace=True)

    # --------------------------------------------------------------------------
    # FASE PENYIMPANAN BERKAS PANDUAN PANGKALAN DATA
    # --------------------------------------------------------------------------
    if not os.path.exists(PATH_DEMO_DIR):
        os.makedirs(PATH_DEMO_DIR)
        
    df_final_report.to_csv(PATH_OUTPUT_PANDUAN, index=False)
    
    print("-" * 90)
    print("🎯 PROSES REKAYASA KATALOG TUNTAS PARIPURNA!")
    print(f"   Berkas panduan unduhan sukses terkunci di:\n   👉 {PATH_OUTPUT_PANDUAN}")
    print(f"   Total target paket unduhan 3C siap jalan: {len(df_final_report)} baris event.")
    print("-" * 90)
    
    # Munculkan 5 baris data teratas langsung di bawah sel Jupyter Bapak untuk verifikasi visual
    display(df_final_report.head())
    print("="*90)

if __name__ == "__main__":
    rekayasa_katalog_master_usgs()

🚀 STARTING: FORENSIC DATA WRANGLING FOR INDONESIA SEISMIC CATALOG
⏳ Memuat file CSV master USGS raksasa ke dalam RAM Mac...
   ✅ Sukses Memuat: Terdeteksi total 83045 baris kejadian gempa global.

⏳ Mengeksekusi pipa penyaringan spasial dan magnitudo regional Indonesia...
   ✅ Filter Berhasil: Ditemukan 23839 kejadian gempa berkekuatan M >= 4.5 di wilayah Indonesia.

⏳ Mentransformasikan parameter data ke format kaku ObsPy ISO-8601...
------------------------------------------------------------------------------------------
🎯 PROSES REKAYASA KATALOG TUNTAS PARIPURNA!
   Berkas panduan unduhan sukses terkunci di:
   👉 /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/Code & Figure demo/katalog_gempa_indonesia_usgs_bmkg.csv
   Total target paket unduhan 3C siap jalan: 23838 baris event.
------------------------------------------------------------------------------------------


,Network,Station,Origin_Time
9,II,KAPI,2001-01-01 12:44:31
18,II,KAPI,2001-01-02 17:21:54
19,II,KAPI,2001-01-02 17:51:34
22,II,KAPI,2001-01-03 20:38:57
23,II,KAPI,2001-01-04 03:50:11


In [ ]:
# -*- coding: utf-8 -*-
import os
import sys
import h5py
import numpy as np
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from obspy.taup import TauPyModel  # Mesin kalkulus interior bumi global tetap aman
from concurrent.futures import ThreadPoolExecutor  # Akselerasi paralel download
from tqdm import tqdm

# ==============================================================================
# 🎛️ PARAMETER ABSOLUT JALUR DIREKTORI DI SSD MAC ANDA
# ==============================================================================
PATH_DEMO_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/Code & Figure demo'
PATH_USGS_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/usgs_katalog/katalog_usgs_master_2001_2025.csv'
PATH_OUTPUT_HDF5 = os.path.join(PATH_DEMO_DIR, 'dataset_indonesia_sejati_3c_700.h5')

# Metrik koordinat sejati sensor permukaan stasiun target (KAPI - Kappang, Sulawesi)
KAPI_LAT = -4.7672
KAPI_LON = 119.7533
KAPI_NET = "II"
KAPI_STA = "KAPI"

# Kunci arsitektur kaku model MCU-Quake 5-20 (7 Detik, 100 Hz, Filter 5-20 Hz)
INPUT_SIZE = 700
FREQ_MIN = 5.0
FREQ_MAX = 20.0
MAX_WORKERS = 12  # Jumlah pipa internet paralel simultan

def hitung_jarak_derajat_pure_math(lat1, lon1, lat2, lon2):
    """Membuka independensi kalkulasi jarak busur derajat bola bumi menggunakan NumPy"""
    try:
        # Konversi koordinat derajat permukaan ke bentuk radian
        r_lat1, r_lon1, r_lat2, r_lon2 = np.radians([lat1, lon1, lat2, lon2])
        
        # Rumus Matematika Spherical Law of Cosines untuk jarak sudut kelengkungan
        cos_delta = (np.sin(r_lat1) * np.sin(r_lat2) + 
                     np.cos(r_lat1) * np.cos(r_lat2) * np.cos(r_lon1 - r_lon2))
        
        # Amankan batas rentang matriks agar tidak memicu nilai imajiner akibat floating-point precision
        cos_delta = np.clip(cos_delta, -1.0, 1.0)
        
        delta_radian = np.arccos(cos_delta)
        return np.degrees(delta_radian)  # Mengembalikan nilai dalam satuan derajat bola bumi
    except Exception:
        return None

def hitung_travel_time_p_arrival(eq_lat, eq_lon, eq_depth_km):
    """Menggunakan model iasp91 untuk menghitung detik perambatan fasa P teoritis"""
    try:
        # 🔥 SOLUSI INDEPENDEN: Memanggil kalkulator matematika murni buatan sendiri
        jarak_derajat = hitung_jarak_derajat_pure_math(eq_lat, eq_lon, KAPI_LAT, KAPI_LON)
        if jarak_derajat is None:
            return None
        
        # Inisialisasi model kecepatan interior bumi iasp91 standar IASPEI
        model_bumi = TauPyModel(model="iasp91")
        
        # Simulasikan penjejakan sinar fasa gelombang kompresional (P)
        arrivals = model_bumi.get_travel_times(
            source_depth_in_km=float(eq_depth_km),
            distance_in_degree=jarak_derajat,
            phase_list=["p", "P"]
        )
        
        if arrivals:
            return arrivals[0].time
        return None
    except Exception:
        return None

def unduh_dan_proses_stasiun_3c_taup(origin_time_str, eq_lat, eq_lon, eq_depth_km):
    """Menarik 3 komponen (N, E, Z) dengan penyelarasan waktu tiba fasa P sejati"""
    client = Client("IRIS")
    
    # Hitung estimasi waktu tempuh gelombang P dari hiposenter ke KAPI
    travel_time_detik = hitung_travel_time_p_arrival(eq_lat, eq_lon, eq_depth_km)
    if travel_time_detik is None:
        return None
        
    try:
        # Sanitasi teks waktu asal patahan
        waktu_origin_bersih = str(origin_time_str).strip().replace(' ', 'T').replace('Z', '')
        t_origin = UTCDateTime(waktu_origin_bersih)
        
        # Geser basis waktu ke Detik Tiba Fasa P sejati di permukaan bumi
        t_p_arrival = t_origin + travel_time_detik
        
        # ----------------------------------------------------------------------
        # JENDELA A: SEISMOGRAM GEMPA SEJATI (LE) -> Berpusat di Kedatangan Gelombang P
        # ----------------------------------------------------------------------
        start_le = t_p_arrival - 2  # Ambil 2 detik sebelum gelombang P menyentuh KAPI
        end_le = t_p_arrival + 5    # Ambil 5 detik setelah gelombang P menyentuh KAPI
        
        st_le = client.get_waveforms(KAPI_NET, KAPI_STA, "*", "BH*", start_le, end_le)
        st_le.detrend("demean")
        st_le.detrend("linear")
        st_le.resample(100.0)
        st_le.filter('bandpass', freqmin=FREQ_MIN, freqmax=FREQ_MAX, corners=4, zerophase=True)
        
        # Ekstrak komponen Z, N, E secara aman dengan index pemotongan [:INPUT_SIZE]
        tr_z_le = st_le.select(component="Z")[0].data[:INPUT_SIZE]
        tr_n_le = st_le.select(component="N")[0].data[:INPUT_SIZE] if st_le.select(component="N") else st_le.select(component="1")[0].data[:INPUT_SIZE]
        tr_e_le = st_le.select(component="E")[0].data[:INPUT_SIZE] if st_le.select(component="E") else st_le.select(component="2")[0].data[:INPUT_SIZE]
        
        # ----------------------------------------------------------------------
        # JENDELA B: AMBIENT NOISE (NO) -> 2 Menit SEBELUM Gelombang P Tiba
        # ----------------------------------------------------------------------
        start_no = t_p_arrival - 120  
        end_no = t_p_arrival - 113
        
        st_no = client.get_waveforms(KAPI_NET, KAPI_STA, "*", "BH*", start_no, end_no)
        st_no.detrend("demean")
        st_no.detrend("linear")
        st_no.resample(100.0)
        st_no.filter('bandpass', freqmin=FREQ_MIN, freqmax=FREQ_MAX, corners=4, zerophase=True)
        
        # Ekstrak komponen Z, N, E untuk noise dengan index pemotongan [:INPUT_SIZE]
        tr_z_no = st_no.select(component="Z")[0].data[:INPUT_SIZE]
        tr_n_no = st_no.select(component="N")[0].data[:INPUT_SIZE] if st_no.select(component="N") else st_no.select(component="1")[0].data[:INPUT_SIZE]
        tr_e_no = st_no.select(component="E")[0].data[:INPUT_SIZE] if st_no.select(component="E") else st_no.select(component="2")[0].data[:INPUT_SIZE]
        
        def bersihkan_array(arr):
            if len(arr) < INPUT_SIZE:
                arr = np.pad(arr, (0, INPUT_SIZE - len(arr)), 'constant')
            if np.std(arr) > 0:
                arr = (arr - np.mean(arr)) / np.std(arr)
            return arr.astype(np.float32)

        # Susun matriks spasial 3C sesuai spesifikasi urutan kaku Zhi Geng: [East, North, Vertical]
        matriks_le = np.stack([bersihkan_array(tr_e_le), bersihkan_array(tr_n_le), bersihkan_array(tr_z_le)], axis=-1)
        matriks_no = np.stack([bersihkan_array(tr_e_no), bersihkan_array(tr_n_no), bersihkan_array(tr_z_no)], axis=-1)
        
        return matriks_le, matriks_no
    except Exception:
        return None

def jalankan_pipeline_rekayasa_taup_3c():
    print("="*90)
    print("🚀 STARTING: MULTI-THREADED PHASE-ALIGNED 3C WAVEFORM DOWNLOADER")
    print("="*90)
    
    if not os.path.exists(PATH_USGS_MASTER):
        print(f"❌ GALAT: Berkas master USGS tidak ditemukan di jalur: {PATH_USGS_MASTER}")
        return

    # 1. Muat Berkas Master USGS dan Terapkan Filter Geografis Indonesia M >= 5.0
    print("⏳ Membaca dan menyaring katalog master USGS...")
    df_master = pd.read_csv(PATH_USGS_MASTER)
    df_master.columns = [col.lower() for col in df_master.columns]
    
    kondisi_indonesia = (df_master['latitude'] >= -11.0) & (df_master['latitude'] <= 6.0) & \
                        (df_master['longitude'] >= 95.0) & (df_master['longitude'] <= 141.0) & \
                        (df_master['mag'] >= 5.0)
    df_indonesia = df_master[kondisi_indonesia].copy()
    
    total_event = len(df_indonesia)
    print(f"   ✅ Filter Berhasil: Mengunci {total_event} event gempa berenergi tinggi.")
    
    koleksi_le, koleksi_no, koleksi_nama = [], [], []
    tugas_list = list(df_indonesia.iterrows())
    
    # 2. Definisikan Fungsi Pembungkus Thread Pekerja
    def tugas_paralel(row_tuple):
        idx, row = row_tuple
        ot, lat, lon = row['time'], row['latitude'], row['longitude']
        
        # Ekstrak nilai kedalaman aman, jika kosong beri nilai default kerak dangkal (15 km)
        depth_km = row['depth'] if 'depth' in row and not pd.isna(row['depth']) else 15.0
        
        hasil = unduh_dan_proses_stasiun_3c_taup(ot, lat, lon, depth_km)
        if hasil is not None:
            return hasil[0], hasil[1], f"{KAPI_NET}_{KAPI_STA}_{str(ot)[:19]}"
        return None

    # 3. Eksekusi Jaringan Paralel Multi-Threading IRIS
    print(f"\n⏳ Membuka {MAX_WORKERS} pipa paralel. Menghitung travel-time & menarik gelombang sejati...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        hasil_generator = list(tqdm(executor.map(tugas_paralel, tugas_list), total=len(tugas_list), desc="Phase-Aligned Downloading"))
        
        for hasil in hasil_generator:
            if hasil is not None:
                mat_le, mat_no, label_name = hasil
                koleksi_le.append(mat_le)
                koleksi_no.append(mat_no)
                koleksi_nama.append(label_name.encode('utf-8'))

    total_sukses = len(koleksi_le)
    print(f"\n   ✅ Sukses Paripurna: Berhasil mengamankan {total_sukses} paket stasiun berfasa P utuh.")
    if total_sukses == 0:
        print("❌ GALAT: Koneksi ditolak server atau data stasiun KAPI kosong pada era tersebut.")
        return

    # --------------------------------------------------------------------------
    # 4. KUNCI MATRIKS KE FILE DATABASE HDF5 TUNGGAL TERKOMPRESI (.h5)
    # --------------------------------------------------------------------------
    print(f"⏳ Membungkus database tensor baru di -> {PATH_OUTPUT_HDF5}")
    with h5py.File(PATH_OUTPUT_HDF5, 'w') as hf:
        array_x = np.concatenate([koleksi_le, koleksi_no], axis=0) 
        array_y = np.concatenate([np.ones(total_sukses, dtype=np.int32), np.zeros(total_sukses, dtype=np.int32)])
        array_name = np.concatenate([koleksi_nama, koleksi_nama])
        
        hf.create_dataset('X', data=array_x, compression="gzip")
        hf.create_dataset('Y', data=array_y, compression="gzip")
        hf.create_dataset('name', data=array_name, compression="gzip")
        
    print(f"🎯 DATABASE BARU SEHAT TERKUNCI: Total {total_sukses * 2} tensor siap uji bebas bias!")
    print("="*90)

if __name__ == "__main__":
    jalankan_pipeline_rekayasa_taup_3c()

🚀 STARTING: MULTI-THREADED PHASE-ALIGNED 3C WAVEFORM DOWNLOADER
⏳ Membaca dan menyaring katalog master USGS...
   ✅ Filter Berhasil: Mengunci 5811 event gempa berenergi tinggi regional Indonesia.

⏳ Membuka 12 pipa paralel internet Mac Anda...
   Menghitung travel-time interior bumi iasp91 & mengunduh gelombang sejati...


/opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)
Phase-Aligned Downloading:  20%|█▉        | 1143/5811 [01:35<06:30, 11.94it/s]


KeyboardInterrupt: 